# Ropedia Academy — B8 · Dynamic & 4D reconstruction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B8.ipynb)

> **Factors a dynamic scene into a canonical model + deformation field, drawing the deformation as a quiver field and the as-rigid-as-possible penalty as a heatmap.**
>
> 把动态场景分解为标准模型 + 形变场，用箭头场画出形变，用热图画出尽可能刚性（ARAP）惩罚。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B8

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt

# 4D = a CANONICAL model (rest scene) + a DEFORMATION field warping time t into it.
canonical = nn.Sequential(nn.Linear(3,64), nn.ReLU(), nn.Linear(64,4))   # shared over time
deform    = nn.Sequential(nn.Linear(4,64), nn.Tanh(), nn.Linear(64,3))   # (x,t) -> delta x
def render_at(x, t):
    dx = deform(torch.cat([x, torch.full_like(x[:, :1], t)], -1))
    return canonical(x + dx), dx

# A grid of points; SAME canonical model, motion grows with time.
g = torch.stack(torch.meshgrid(torch.linspace(-1,1,12), torch.linspace(-1,1,12),
                               indexing='ij'), -1).reshape(-1,3)
g = torch.cat([g[:,:2], torch.zeros(len(g),1)], -1).requires_grad_()
_, d1 = render_at(g, 1.0)
print("appearance shared across time; only the deformation field varies")

fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
xn, dn = g.detach().numpy(), d1.detach().numpy()
ax[0].quiver(xn[:,0], xn[:,1], dn[:,0], dn[:,1], angles='xy'); ax[0].set_title("deformation field at t=1")
nbr = torch.cdist(g, g).topk(4, largest=False).indices            # as-rigid-as-possible
arap = (d1[nbr] - d1[:, None]).pow(2).mean(-1).mean(-1).detach()
sc = ax[1].scatter(xn[:,0], xn[:,1], c=arap, cmap='plasma'); plt.colorbar(sc, ax=ax[1])
ax[1].set_title("local-rigidity (ARAP) penalty"); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B8
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks